Loading the raw counts and metadata

In [1]:
import pandas as pd

df_raw_cols = pd.read_csv('data/GSE189149_raw_counts_GRCh38.p13_NCBI(1).tsv.gz', sep='\t', index_col=0, nrows=0).columns.tolist()
metadata_csv = pd.read_csv('data/GSE189149_sample_metadata.csv')
print(df_raw_cols == metadata_csv['gsm_id'].tolist())

True


In [2]:
#imports
import pandas as pd
import numpy as np
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

In [3]:
#load counts + metadata
counts = pd.read_csv('data/GSE189149_raw_counts_GRCh38.p13_NCBI(1).tsv.gz',
                      sep='\t', index_col=0)
metadata = pd.read_csv('data/GSE189149_sample_metadata.csv', index_col='gsm_id')

print(counts.shape)
print(metadata.shape)

(39376, 48)
(48, 2)


In [4]:
#fix the orientation, then check alignment
counts = counts.T
print(counts.shape)  #should now say (48, 39376)

assert list(counts.index) == list(metadata.index), "order's off, fix before continuing"
print("alignment ok")

(48, 39376)
alignment ok


In [5]:
#drop genes with barely any counts across all samples
gene_totals = counts.sum(axis=0)  #total counts per gene, summed across all 48 samples
keep = gene_totals >= 10  #keep genes with at least 10 total reads overall

counts_filtered = counts.loc[:, keep]
print(f"{counts.shape[1]} genes before, {counts_filtered.shape[1]} after")

39376 genes before, 24745 after


In [6]:
#build the dataset + fit
inference = DefaultInference(n_cpus=4)  

dds = DeseqDataSet(
    counts=counts_filtered,
    metadata=metadata,
    design="~condition",
    inference=inference
)
dds.deseq2()

Using None as control genes, passed at DeseqDataSet initialization


/home/ethan-xiao/miniconda3/lib/python3.14/functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
Fitting size factors...
... done in 0.02 seconds.

Fitting dispersions...
... done in 1.71 seconds.

Fitting dispersion trend curve...
... done in 0.21 seconds.

Fitting MAP dispersions...
... done in 2.19 seconds.

Fitting LFCs...
... done in 1.16 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 52 outlier genes.

Fitting dispersions...
... done in 0.01 seconds.

Fitting MAP dispersions...
... done in 0.01 seconds.

Fitting LFCs...
... done in 0.01 seconds.



In [7]:
#run the test
stat_res = DeseqStats(dds, contrast=["condition", "allergic", "control"], inference=inference)
stat_res.summary()
results_df = stat_res.results_df

Running Wald tests...


Log2 fold change & Wald test p-value: condition allergic vs control
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
GeneID                                                                        
100287102     1.886889        0.368569  0.403436  0.913575  0.360940  0.705667
653635      343.577440        0.025328  0.104049  0.243426  0.807675  0.936396
102466751     7.156201        0.184946  0.262274  0.705162  0.480710  0.788174
107985730     0.519661        0.537260  0.746528  0.719678  0.471723       NaN
100996442    17.308118        0.278813  0.170690  1.633452  0.102374  0.428643
...                ...             ...       ...       ...       ...       ...
4541       1814.715382       -0.020675  0.213033 -0.097050  0.922687  0.978370
4556        141.506925       -0.080796  0.185266 -0.436107  0.662759  0.879135
4519       6392.046730        0.098560  0.241322  0.408416  0.682968  0.888562
4576         25.652110       -0.073585  0.215511 -0.341443  0.7

... done in 0.49 seconds.



In [8]:
#isolate isg15
isg15_row = results_df.loc['9636']
print(isg15_row)

baseMean          3568.810290
log2FoldChange      -0.925921
lfcSE                0.401111
stat                -2.308393
pvalue               0.020977
padj                 0.245340
Name: 9636, dtype: float64


In [9]:
results_df.to_csv('data/deseq2_results_GSE189149.csv')

- RNA-seq differential expression (DESeq2) run on GSE189149 (48 adolescent samples, allergic vs. control), naive/activated CD4+ T cells.
- 39,376 genes before low-count filtering (≥10 total reads), 24,745 after.
- ISG15 (Entrez 9636): log2FC = -0.926, p = 0.021, padj = 0.245 (closely matches Imran et al.'s independently published result, log2FC = -0.931, p = 0.021, on the same samples)
- Full results saved to deseq2_results_GSE189149.csv, used downstream in notebook 10's cross-omics candidate ranking.

Feeds into notebook 10 (RNA-seq cross-validation of methylation candidates).